In [ ]:

dbutils.widgets.text("catalog", "", "Catalog Name")
dbutils.widgets.text("schema", "", "Schema Name")
dbutils.widgets.text("warehouse", "", "Warehouse")
dbutils.widgets.text("govern_table", "", "AI Governance output table")

In [ ]:
from uuid import uuid4

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
warehouse = dbutils.widgets.get("warehouse")
govern_table = dbutils.widgets.get("govern_table")

run_id = f"{uuid4()}"

In [ ]:
from databricks.sdk.service.dashboards import GenieSpace
from governer.utils import task_genie

def govern_table_for_pii(space: GenieSpace, table: str):
    table_pii_govern_prompt=f"""
You are a data governance assistant for a Databricks workspace.

Your task is to analyze the schema of the table named {table} and identify columns that are likely to contain Personally Identifiable Information (PII).

Use the column names, data types, tags, and any available metadata or sample values to make your assessment.

IMPORTANT:

Only evaluate columns that DO NOT already have the tag "pii".
Ignore any column where the tag "pii" is already present.

PII includes, but is not limited to:

- Direct identifiers (e.g., name, email, phone number, national ID, passport number)
- Quasi-identifiers (e.g., date of birth, address, zip code, IP address)
- Sensitive personal data (e.g., financial details, health information, biometric data)

For each relevant column, assign a confidence level based on your assessment:

1 -  Very likely contains PII  
0.7 - Possibly contains PII  
0.3 - Unlikely but could contain PII depending on context  

Additionally, for each detected column, generate the corresponding SQL statement to apply the "pii" tag using Databricks Unity Catalog syntax.

Assume tagging is done at the column level using:
SET TAG ON COLUMN {table}.<column_name> (`pii`=`<suspected_pii_type>`)

Output requirements:

Return ONLY a JSON array.

Each element must include:

- column_name
- suspected_pii_type (e.g., email, phone_number, name, address, etc.)
- confidence (HIGH, MEDIUM, LOW)
- sql_statement (SQL string to apply the pii tag to that column)

Do not include explanations or additional text.
Do not include columns that are clearly non-PII.
Do not include columns that already have the "pii" tag.

Example output:

[
{{
  "column_name": "email_address",
  "suspected_pii_type": "email",
  "confidence": 1,
  "sql_statement": "SET TAG ON COLUMN {table}.email_address (`pii`=`email`)"
}},
{{
  "column_name": "zip_code",
  "suspected_pii_type": "location",
  "confidence": 0.7,
  "sql_statement": "SET TAG ON COLUMN {table}.zip_code (`pii`=`location`)"
}}
]
]
    """
    return task_genie(space=space, task=table_pii_govern_prompt)

In [ ]:
from governer.utils import get_valid_govern_tables

spark.sql(f"use catalog {catalog}")
spark.sql(f"use schema {schema}")

tables = get_valid_govern_tables(catalog=catalog, schema=schema, tables=spark.catalog.listTables())



In [ ]:
from typing import List
from governer.utils import AiResponse, AiResponseWithEval
import json

def parse_table_pii_results(result: AiResponse) -> List[AiResponseWithEval]:
    bad = [AiResponseWithEval(result.prompt, result.text, result.success, result.error, 0)]
    if not result.success:
        return bad
    
    try:
        res = []
        parsed = json.load(result.text)
        for entry in parsed:
            res.append(AiResponseWithEval(result.prompt, entry["sql_statement"], True, None, entry["confidence"]))
    except Exception:
        return bad
    

In [ ]:
from governer.utils import trash_genie_workspace, get_genie_workspace, GovernStatus
from governer.schemas.governance.ai_govern_table import ai_govern_table
import datetime

genie_space_title="Genie Governer Space PII"
genie_space = get_genie_workspace(space_title=genie_space_title, warehouse_id=warehouse, tables=tables)

res = []

for table in tables:
    response = govern_table_for_pii(space=genie_space, table=table)
    print(f"table={table} response = {response.text}")
    parsed = parse_table_pii_results(result=response)
    for resp in parsed:
        res.append({
            "record_id": f"{uuid4()}",
            "run_id": run_id, 
            "table_name": table, 
            "govern_success": resp.success, 
            "govern_error": resp.error,
            "evaluation": resp.eval,
            "sql": resp.text, 
            "status": GovernStatus.GENERATED.name, 
            "timestamp": datetime.datetime.now()
        })
        


trash_genie_workspace(genie_space)
govern = spark.createDataFrame(data=res,schema=ai_govern_table)
govern.write.mode("append").saveAsTable(govern_table)
